# Microsoft GraphRAG: Melhorando Retrieval-Augmented Generation com gráficos de conhecimento

## Visão Geral

Microsoft GraphRAG é um sistema avançado Retrieval-Augmented Generation (RAG) que integra gráficos de conhecimento para melhorar o desempenho de grandes modelos de linguagem (LLMs). Desenvolvido pela Microsoft Research, GraphRAG aborda limitações nas abordagens tradicionais RAG usando gráficos de conhecimento gerados por LLM para melhorar a análise de documentos e melhorar a qualidade da resposta.

## Motivação

Os sistemas tradicionais RAG muitas vezes lutam com consultas complexas que requerem sintetizar informações de fontes díspares. GraphRAG visa:
Conecte informações relacionadas entre conjuntos de dados.
Melhorar a compreensão de conceitos semânticos.
Melhorar o desempenho em tarefas globais de criação de sentidos.

## Componentes-chave

Geração de Gráficos de Conhecimento: Construi gráficos com entidades como nós e relações como bordas.
Detecção da Comunidade: Identifica clusters de entidades relacionadas dentro do gráfico.
Resumo: Gera resumos para cada comunidade para fornecer contexto para LLMs.
Processamento de Consultas: Utiliza estes resumos para aumentar a capacidade da LLM de responder a perguntas complexas.
## Detalhes do método
Fase de indexação


Text Chunking: Divide textos fonte em chunks gerenciáveis.
Extração de elementos: Utiliza LLMs para identificar entidades e relacionamentos.
Construção de Gráficos: Compila um gráfico dos elementos extraídos.
Detecção da Comunidade: Aplica algoritmos como Leiden para encontrar comunidades.
Resumição comunitária: Cria resumos para cada comunidade.

Fase de Consulta


Geração de Resposta Local: Utiliza resumos comunitários para gerar respostas preliminares.
Síntese de Resposta Global: Combina respostas locais para formar uma resposta abrangente.


## Benefícios de GraphRAGGraphRAG é uma ferramenta poderosa que aborda algumas das principais limitações do modelo RAG de linha de base. Ao contrário do modelo padrão RAG, GraphRAG se destaca na identificação de conexões entre partes díspares de informação e insights de desenho deles. Isso o torna uma escolha ideal para usuários que precisam extrair insights de grandes coleções de dados ou documentos que são difíceis de resumir. Ao alavancar sua arquitetura avançada baseada em gráficos, o GraphRAG é capaz de fornecer uma compreensão holística de conceitos semânticos complexos, tornando-se uma ferramenta inestimável para quem precisa encontrar informações de forma rápida e precisa. Quer você seja pesquisador, analista ou apenas alguém que precisa ficar informado, o GraphRAG pode ajudá-lo a conectar os pontos e descobrir novos insights.

## Conclusão
O Microsoft GraphRAG representa um avanço significativo na geração aumentada de recuperação, particularmente para tarefas que exigem uma compreensão global de conjuntos de dados. Ao incorporar gráficos de conhecimento, oferece melhor desempenho, tornando-o ideal para recuperação e análise de informações complexas.

Para aqueles experientes com sistemas RAG básicos, GraphRAG oferece uma oportunidade para explorar soluções mais sofisticadas, embora não seja necessário para todos os casos de uso.
A Retrieval Aumented Generation (RAG) é muitas vezes executada através de chunks de textos longos, criando um texto incorporado para cada bloco, e recuperando chunks para incluir no contexto de geração LLM baseado em uma pesquisa de similaridade contra a consulta. Esta abordagem funciona bem em muitos cenários, e com rapidez e custos convincentes trade-offs, mas nem sempre lida bem em cenários onde uma compreensão detalhada do texto é necessária.

GraphRag ( [microsoft.github.io/graphrag](https://microsoft.github.io/graphrag/) )

<div style="text-align: center;">

<img src="../images/Microsoft_GraphRag.svg" alt="adaptive retrieval" style="width:100%; height:auto;">
</div>

Para executar este notebook você pode usar OpenAI API chave ou Azure OpenAI chave.
Crie um arquivo `.env` e preencha as credenciais para sua implantação de IA aberta OpenAI ou Azure. O código a seguir carrega essas variáveis de ambiente e configura nosso cliente IA.


In [ ]:
AZURE_OPENAI_API_KEY=""
AZURE_OPENAI_ENDPOINT=""
GPT4O_MODEL_NAME="gpt-4o"
TEXT_EMBEDDING_3_LARGE_DEPLOYMENT_NAME=""
AZURE_OPENAI_API_VERSION="2024-06-01"

OPENAI_API_KEY=""

In [ ]:
%pip install graphrag

# Instalação e Importações do Pacote
A célula abaixo instala todos os pacotes necessários para executar este notebook.


In [ ]:
# Install required packages
!pip install beautifulsoup4 openai python-dotenv pyyaml

Instalação do Pacote

A célula abaixo instala todos os pacotes necessários para executar este notebook. Se você estiver executando este notebook em um novo ambiente, execute esta célula primeiro para garantir que todas as dependências estejam instaladas.

In [ ]:
# Install required packages
!pip install openai python-dotenv

In [1]:
from dotenv import load_dotenv
import os
load_dotenv()
from openai import AzureOpenAI, OpenAI

AZURE=True #Change to False to use OpenAI
if AZURE:
    AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
    AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
    GPT4O_DEPLOYMENT_NAME = os.getenv("GPT4O_MODEL_NAME")
    TEXT_EMBEDDING_3_LARGE_NAME = os.getenv("TEXT_EMBEDDING_3_LARGE_DEPLOYMENT_NAME")
    AZURE_OPENAI_API_VERSION = os.getenv("AZURE_OPENAI_API_VERSION")
    oai = AzureOpenAI(azure_endpoint=AZURE_OPENAI_ENDPOINT, api_key=AZURE_OPENAI_API_KEY, api_version=AZURE_OPENAI_API_VERSION)
else:
    OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
    oai = OpenAI(api_key=OPENAI_API_KEY)


Vamos começar por enviar uma mensagem. O artigo da Wikipédia sobre Elon Musk

In [2]:
import requests
from bs4 import BeautifulSoup

url = "https://en.wikipedia.org/wiki/Elon_Musk"  # Replace with the URL of the web page you want to scrape
response = requests.get(url)
soup = BeautifulSoup(response.text, "html.parser")

if not os.path.exists('data'): 
    os.makedirs('data')

if not os.path.exists('data/elon.md'):
    elon = soup.text.split('\nSee also')[0]
    with open('data/elon.md', 'w', encoding='utf-8') as f:
        f.write(elon)
else:
    with open('data/elon.md', 'r') as f:
        elon = f.read()


GraphRag tem um conjunto conveniente de comandos CLI que podemos usar. Vamos começar configurando o sistema e, em seguida, executar a operação de indexação. A indexação com GraphRag é um processo muito mais longo, e um que custa significativamente mais, uma vez que ao invés de apenas calcular embutimentos, GraphRag faz muitas chamadas LLM para analisar o texto, extrair entidades e construir o gráfico. Mas é uma despesa única.

In [ ]:
import yaml

if not os.path.exists('data/graphrag'):
    !python -m graphrag.index --init --root data/graphrag

with open('data/graphrag/settings.yaml', 'r') as f:
    settings_yaml = yaml.load(f, Loader=yaml.FullLoader)
settings_yaml['llm']['model'] = "gpt-4o"
settings_yaml['llm']['api_key'] = AZURE_OPENAI_API_KEY if AZURE else OPENAI_API_KEY
settings_yaml['llm']['type'] = 'azure_openai_chat' if AZURE else 'openai_chat'
settings_yaml['embeddings']['llm']['api_key'] = AZURE_OPENAI_API_KEY if AZURE else OPENAI_API_KEY
settings_yaml['embeddings']['llm']['type'] = 'azure_openai_embedding' if AZURE else 'openai_embedding'
settings_yaml['embeddings']['llm']['model'] = TEXT_EMBEDDING_3_LARGE_NAME if AZURE else 'text-embedding-3-large'
if AZURE:
    settings_yaml['llm']['api_version'] = AZURE_OPENAI_API_VERSION
    settings_yaml['llm']['deployment_name'] = GPT4O_DEPLOYMENT_NAME
    settings_yaml['llm']['api_base'] = AZURE_OPENAI_ENDPOINT
    settings_yaml['embeddings']['llm']['api_version'] = AZURE_OPENAI_API_VERSION
    settings_yaml['embeddings']['llm']['deployment_name'] = TEXT_EMBEDDING_3_LARGE_NAME
    settings_yaml['embeddings']['llm']['api_base'] = AZURE_OPENAI_ENDPOINT

with open('data/graphrag/settings.yaml', 'w') as f:
    yaml.dump(settings_yaml, f)

if not os.path.exists('data/graphrag/input'):
    os.makedirs('data/graphrag/input')
    !cp data/elon.md data/graphrag/input/elon.txt
    !python -m graphrag.index --root ./data/graphrag

Você deve obter uma saída:
□ Todos os fluxos de trabalho concluídos com sucesso.

Para consultar o Gráfico Rag usaremos seu CLI novamente, certificando-se de configurá-lo com um comprimento de contexto equivalente ao que usamos em nossa pesquisa de incorporação.

In [6]:
import subprocess
import re
DEFAULT_RESPONSE_TYPE = 'Summarize and explain in 1-2 paragraphs with bullet points using at most 300 tokens'
DEFAULT_MAX_CONTEXT_TOKENS = 10000

def remove_data(text):
    return re.sub(r'\[Data:.*?\]', '', text).strip()


def ask_graph(query,method):
    env = os.environ.copy() | {
      'GRAPHRAG_GLOBAL_SEARCH_MAX_TOKENS': str(DEFAULT_MAX_CONTEXT_TOKENS),
    }
    command = [
      'python', '-m', 'graphrag.query',
      '--root', './data/graphrag',
      '--method', method,
      '--response_type', DEFAULT_RESPONSE_TYPE,
      query,
    ]
    output = subprocess.check_output(command, universal_newlines=True, env=env, stderr=subprocess.DEVNULL)
    return remove_data(output.split('Search Response: ')[1])

GrpahRag oferece 2 tipos de pesquisa:
1. Global Search for raciocine about holistic questions about the corpus, alavancando os resumos da comunidade.
2. Busca local para raciocinar sobre entidades específicas por fanning-out para seus vizinhos e conceitos associados.

Vamos verificar a pesquisa local:

In [8]:
from IPython.display import Markdown
local_query="What and how many companies and subsidieries founded by Elon Musk"
local_result = ask_graph(local_query,'local')

Markdown(local_result)

Elon Musk has founded several companies and subsidiaries across various industries. Here's a summary:

- **SpaceX**: Founded in 2002, SpaceX is a private aerospace manufacturer and space transportation company. Musk serves as the CEO and chief engineer .

- **Tesla, Inc.**: Although not originally founded by Musk, he became an early investor and later the CEO and product architect, significantly shaping its direction .

- **Neuralink**: Co-founded by Musk, this company focuses on developing brain-machine interfaces to enhance human-computer interaction .

- **The Boring Company**: Founded by Musk, it specializes in tunnel construction and innovative transportation solutions .

- **X.com/PayPal**: Musk co-founded X.com, which later became PayPal after merging with Confinity .

- **Zip2**: Co-founded with his brother Kimbal, this was Musk's first venture, later acquired by Compaq .

- **SolarCity**: Co-created by Musk, it was later acquired by Tesla and rebranded as Tesla Energy .

- **xAI**: Founded in 2023, this company focuses on artificial intelligence research .

- **OpenAI**: Co-founded by Musk, this nonprofit organization is dedicated to AI research .

In total, Musk has founded or co-founded at least nine companies and subsidiaries.

In [9]:
global_query="What are the major accomplishments of Elon Musk?"
global_result = ask_graph(global_query,'global')

Markdown(global_result)

Elon Musk has achieved significant accomplishments across various industries, demonstrating his influence and innovation:

- **Space Exploration**: Founder, CEO, and chief engineer of SpaceX, Musk has propelled the company to the forefront of space exploration and satellite deployment, establishing it as a leading spaceflight services provider .

- **Automotive Industry**: As CEO of Tesla, Musk has driven the company to the forefront of electric vehicles and sustainable energy, significantly impacting the automotive industry with innovations in electric cars and energy solutions .

- **Online Payments**: Co-founded X.com, which evolved into PayPal, revolutionizing online transactions and becoming a major player in the online payment industry .

- **Neural Technology**: Co-founded Neuralink, focusing on advancing brain-machine interface technology to enhance the connection between the human brain and computers .

- **Infrastructure**: Founded The Boring Company, specializing in tunnel construction to reduce traffic congestion through innovative underground transportation systems .

![](https://europe-west1-rag-techniques-views-tracker.cloudfunctions.net/rag-techniques-tracker?notebook=all-rag-techniques--microsoft-graphrag)